# 출퇴근 스트레스 파이프라인 (CSV 전용, E드라이브)

- CSV 파일만 읽도록 구성했습니다.
- 점수는 **높을수록 스트레스가 큼**을 의미합니다.
- 코드가 순서대로 보이도록 번호를 나눴습니다.

In [1]:
# 1) 라이브러리 불러오기
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations


In [2]:
# 2) 입력/출력 경로 설정
# 현장에서 아래 경로만 바꿔서 사용하면 됩니다.
INPUT_DIR = Path(r"C:\Users\yiho1\Downloads\20230101_monthly")
OUTPUT_DIR = Path(r"C:\Users\yiho1\Documents\Codex\2026-04-24-new-chat\output_from_ipynb")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_DIR.exists():
    raise FileNotFoundError(f"입력 폴더가 없습니다: {INPUT_DIR}")

PermissionError: [WinError 5] 액세스가 거부되었습니다: 'C:\\Users\\yiho1'

In [ ]:
# 3) 폴더 내 CSV 파일 목록 확인
csv_files = sorted(INPUT_DIR.glob("*.csv"))
print(f"발견된 CSV 파일 수: {len(csv_files)}")
for f in csv_files:
    print(f"  {f.name}")

In [ ]:
# 4) 기본 설정값
USE_COLS = [
    "start_dt",
    "arv_dt",
    "start_emd",
    "arv_emd",
    "start_arv_place_type",
    "mvmn_time_sum",
    "popl_cnt"
]

# 유형 우선순위: 1순위부터 누적, 60% 이상이면 중단
PRIORITY_GROUPS = [
    ["HW", "WH"],
    ["HE", "EH"],
    ["WE", "EW"],
    ["HH", "WW", "EE"]
]

TARGET_RATIO = 0.60
YEAR_FROM, YEAR_TO = 2023, 2023
FILTER_H_INCLUDED = True
FEATURE_COLS = ["총이동인구수", "평균이동시간", "총이동시간"]

# 문헌 기반 초기 가중치 (합=1)
LITERATURE_WEIGHTS = {
    "유입인구": 0.45,
    "평균이동시간": 0.35,
    "총이동시간": 0.20
}

# 외부 타깃 컬럼이 있을 때만 회귀기반 가중치 사용
# 예: REG_TARGET_COL = "external_stress_index"
REG_TARGET_COL = None


In [ ]:
# 5) 공통 유틸 함수
def minmax(s: pd.Series) -> pd.Series:
    if s.max() == s.min():
        return pd.Series(0.0, index=s.index)
    return (s - s.min()) / (s.max() - s.min())

def normalize_weights(w: dict) -> dict:
    total = sum(w.values())
    if total == 0:
        n = len(w)
        return {k: 1 / n for k in w}
    return {k: v / total for k, v in w.items()}

def rank_spearman(df_a, df_b, key="자치구코드", col="출퇴근스트레스점수"):
    a = df_a[[key, col]].rename(columns={col: "a"})
    b = df_b[[key, col]].rename(columns={col: "b"})
    m = a.merge(b, on=key, how="inner")
    if len(m) < 2:
        return np.nan
    return m["a"].rank().corr(m["b"].rank(), method="pearson")

def topk_overlap(df_a, df_b, k=5, key="자치구코드", col="출퇴근스트레스점수"):
    sa = set(df_a.sort_values(col, ascending=False).head(k)[key])
    sb = set(df_b.sort_values(col, ascending=False).head(k)[key])
    return len(sa & sb) / k if k > 0 else np.nan


In [ ]:
# 6) 전처리 + 우선순위 필터 함수
def preprocess_movement(raw_df: pd.DataFrame) -> pd.DataFrame:
    df = raw_df[USE_COLS].copy()

    df["start_dt"] = pd.to_datetime(df["start_dt"], errors="coerce")
    df["arv_dt"] = pd.to_datetime(df["arv_dt"], errors="coerce")
    df["start_emd"] = df["start_emd"].astype(str).str.strip()
    df["arv_emd"] = df["arv_emd"].astype(str).str.strip()
    df["start_arv_place_type"] = df["start_arv_place_type"].astype(str).str.upper().str.strip()
    df["mvmn_time_sum"] = pd.to_numeric(df["mvmn_time_sum"], errors="coerce")
    df["popl_cnt"] = pd.to_numeric(df["popl_cnt"], errors="coerce")

    df = df.dropna(subset=["start_dt", "mvmn_time_sum", "popl_cnt"]).copy()
    df = df[df["start_dt"].dt.year.between(YEAR_FROM, YEAR_TO)].copy()
    df["연도"] = df["start_dt"].dt.year

    df["출발_자치구코드"] = df["start_emd"].str[:5]
    df["도착_자치구코드"] = df["arv_emd"].str[:5]

    if FILTER_H_INCLUDED:
        df = df[df["start_arv_place_type"].str.contains("H", na=False)].copy()

    return df

def select_place_types_by_priority(df: pd.DataFrame, target_ratio=0.60):
    total_pop = df["popl_cnt"].sum()
    selected_types = []
    selected_pop = 0.0

    for i, group in enumerate(PRIORITY_GROUPS, start=1):
        group_pop = df.loc[df["start_arv_place_type"].isin(group), "popl_cnt"].sum()
        selected_types.extend(group)
        selected_pop += group_pop
        ratio = selected_pop / total_pop if total_pop > 0 else 0.0
        print(f"[{i}순위] {group} 추가 | 누적 커버리지: {ratio:.2%}")
        if ratio >= target_ratio:
            break

    filtered_df = df[df["start_arv_place_type"].isin(selected_types)].copy()
    coverage = selected_pop / total_pop if total_pop > 0 else 0.0
    return filtered_df, selected_types, coverage


In [ ]:
# 7) 피처 생성 함수
def make_features(df: pd.DataFrame, group_col="도착_자치구코드") -> pd.DataFrame:
    f = (
        df.groupby(group_col, dropna=False)
        .agg(
            총이동인구수=("popl_cnt", "sum"),
            평균이동시간=("mvmn_time_sum", "mean")
        )
        .reset_index()
        .rename(columns={group_col: "자치구코드"})
    )
    f["총이동시간"] = f["총이동인구수"] * f["평균이동시간"]
    return f

def make_features_by_year(df: pd.DataFrame, group_col="도착_자치구코드") -> pd.DataFrame:
    fy = (
        df.groupby(["연도", group_col], dropna=False)
        .agg(
            총이동인구수=("popl_cnt", "sum"),
            평균이동시간=("mvmn_time_sum", "mean")
        )
        .reset_index()
        .rename(columns={group_col: "자치구코드"})
    )
    fy["총이동시간"] = fy["총이동인구수"] * fy["평균이동시간"]
    return fy


In [ ]:
# 8) 가중치 산정 함수 (PCA / CRITIC / 회귀)
def pca_weights(feature_df: pd.DataFrame) -> dict:
    X = feature_df[FEATURE_COLS].astype(float)
    X = (X - X.mean()) / X.std(ddof=0).replace(0, np.nan)
    X = X.fillna(0.0)

    cov = np.cov(X.values, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    pc1 = eigvecs[:, np.argmax(eigvals)]
    load = np.abs(pc1)

    w = {
        "유입인구": load[0],
        "평균이동시간": load[1],
        "총이동시간": load[2]
    }
    return normalize_weights(w)

def critic_weights(feature_df: pd.DataFrame) -> dict:
    Z = feature_df[FEATURE_COLS].astype(float)
    Z = (Z - Z.min()) / (Z.max() - Z.min())
    Z = Z.fillna(0.0)

    std = Z.std(ddof=0)
    corr = Z.corr().fillna(0.0)

    cvals = {}
    for c in FEATURE_COLS:
        cvals[c] = std[c] * (1 - corr[c]).sum()

    w = {
        "유입인구": cvals["총이동인구수"],
        "평균이동시간": cvals["평균이동시간"],
        "총이동시간": cvals["총이동시간"]
    }
    return normalize_weights(w)

def regression_weights(feature_df: pd.DataFrame, target_col: str | None) -> dict | None:
    if not target_col or target_col not in feature_df.columns:
        return None

    d = feature_df.dropna(subset=FEATURE_COLS + [target_col]).copy()
    if len(d) < 5:
        return None

    X = d[FEATURE_COLS].astype(float)
    y = pd.to_numeric(d[target_col], errors="coerce")
    valid = y.notna()
    X, y = X[valid], y[valid]
    if len(X) < 5:
        return None

    Xz = (X - X.mean()) / X.std(ddof=0).replace(0, np.nan)
    Xz = Xz.fillna(0.0)
    yz = (y - y.mean()) / (y.std(ddof=0) if y.std(ddof=0) != 0 else 1.0)

    beta, *_ = np.linalg.lstsq(Xz.values, yz.values, rcond=None)
    beta = np.abs(beta)

    w = {
        "유입인구": beta[0],
        "평균이동시간": beta[1],
        "총이동시간": beta[2]
    }
    return normalize_weights(w)


In [ ]:
# 9) 점수 계산 + 통합 파이프라인 함수
def score_with_weights(feature_df: pd.DataFrame, weights: dict, model_name: str) -> pd.DataFrame:
    d = feature_df.copy()
    d["유입인구_norm"] = minmax(d["총이동인구수"])
    d["평균이동시간_norm"] = minmax(d["평균이동시간"])
    d["총이동시간_norm"] = minmax(d["총이동시간"])

    w = normalize_weights(weights)
    d["출퇴근스트레스점수"] = (
        d["유입인구_norm"] * w["유입인구"]
        + d["평균이동시간_norm"] * w["평균이동시간"]
        + d["총이동시간_norm"] * w["총이동시간"]
    ) * 100

    d["모형"] = model_name
    d["가중치_유입인구"] = w["유입인구"]
    d["가중치_평균이동시간"] = w["평균이동시간"]
    d["가중치_총이동시간"] = w["총이동시간"]

    return d.sort_values("출퇴근스트레스점수", ascending=False).reset_index(drop=True)

def run_pipeline_all(raw_df: pd.DataFrame):
    d = preprocess_movement(raw_df)
    d_sel, selected_types, coverage = select_place_types_by_priority(d, TARGET_RATIO)
    f = make_features(d_sel, "도착_자치구코드")

    w_lit = normalize_weights(LITERATURE_WEIGHTS)
    w_pca = pca_weights(f)
    w_critic = critic_weights(f)
    w_reg = regression_weights(f, REG_TARGET_COL)

    results = [
        score_with_weights(f, w_lit, "문헌기반"),
        score_with_weights(f, w_pca, "PCA기반"),
        score_with_weights(f, w_critic, "CRITIC기반")
    ]
    if w_reg is not None:
        results.append(score_with_weights(f, w_reg, "회귀기반"))

    all_scores = pd.concat(results, ignore_index=True)

    base = all_scores[all_scores["모형"] == "문헌기반"].copy()
    sensitivity_df = pd.DataFrame([
        {
            "비교모형": m,
            "스피어만순위상관(문헌대비)": rank_spearman(base, all_scores[all_scores["모형"] == m].copy()),
            "Top5중복률(문헌대비)": topk_overlap(base, all_scores[all_scores["모형"] == m].copy(), k=5)
        }
        for m in all_scores["모형"].unique()
    ])

    fy = make_features_by_year(d_sel, "도착_자치구코드")
    yearly_parts = []
    for y in sorted(fy["연도"].unique()):
        fy_y = fy[fy["연도"] == y].drop(columns=["연도"]).copy()
        sy = score_with_weights(fy_y, w_lit, f"문헌기반_{y}")
        sy["연도"] = y
        yearly_parts.append(sy[["연도", "자치구코드", "출퇴근스트레스점수"]])
    yearly_scores_df = pd.concat(yearly_parts, ignore_index=True)

    stability_rows = []
    years = sorted(yearly_scores_df["연도"].unique())
    for y1, y2 in combinations(years, 2):
        a = yearly_scores_df[yearly_scores_df["연도"] == y1]
        b = yearly_scores_df[yearly_scores_df["연도"] == y2]
        rho = rank_spearman(a, b)
        stability_rows.append({"연도쌍": f"{y1}-{y2}", "스피어만순위상관": rho})
    stability_df = pd.DataFrame(stability_rows)

    meta_df = pd.DataFrame([{
        "분석기간": f"{YEAR_FROM}~{YEAR_TO}",
        "H포함필터": FILTER_H_INCLUDED,
        "선정유형": ", ".join(selected_types),
        "커버리지": coverage
    }])

    return {
        "메타정보": meta_df,
        "전처리후_선정데이터": d_sel,
        "모형별_점수결과": all_scores,
        "민감도분석": sensitivity_df,
        "연도안정성분석": stability_df,
        "연도별점수_문헌기반": yearly_scores_df
    }


In [ ]:
# 10) 단일 파일 실행 (단건 확인용 - 배치 실행은 11번 셀 사용)
# outputs = run_pipeline_all(raw_df)
# display(outputs["모형별_점수결과"].head())

In [ ]:
# 11) 월별 파일 배치 실행 -> 연도 통합 결과 생성
def run_monthly_folder(input_dir, file_pattern="*.csv", output_dir=None):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir) if output_dir else OUTPUT_DIR / "monthly_batch"
    output_dir.mkdir(parents=True, exist_ok=True)

    files = sorted(input_dir.glob(file_pattern))
    monthly_rows = []
    monthly_scores_all = []

    for fp in files:
        try:
            dfm = pd.read_csv(fp, sep=None, engine="python", encoding="utf-8-sig")
            dfm.columns = dfm.columns.astype(str).str.strip()
            if len(dfm.columns) == 1 and "|" in dfm.columns[0]:
                dfm = pd.read_csv(fp, sep="|", engine="python", encoding="utf-8-sig")
                dfm.columns = dfm.columns.astype(str).str.strip()

            out = run_pipeline_all(dfm)
            sc = out["모형별_점수결과"].copy()

            digits = "".join(ch for ch in fp.stem if ch.isdigit())
            ym = digits[:6] if len(digits) >= 6 else None
            if ym:
                y = int(ym[:4])
                m = int(ym[4:6])
            else:
                dtmp = out["전처리후_선정데이터"]["start_dt"] if len(out["전처리후_선정데이터"]) else pd.Series(dtype="datetime64[ns]")
                y = int(dtmp.dt.year.mode().iloc[0]) if len(dtmp) else np.nan
                m = int(dtmp.dt.month.mode().iloc[0]) if len(dtmp) else np.nan

            if len(sc):
                sc["연도"] = y
                sc["월"] = m
                sc["원본파일"] = fp.name
                monthly_scores_all.append(sc)

            coverage = float(out["메타정보"]["커버리지"].iloc[0]) if len(out["메타정보"]) else np.nan
            rows_after = len(out["전처리후_선정데이터"])
            monthly_rows.append({
                "파일명": fp.name,
                "원본행수": len(dfm),
                "전처리후행수": rows_after,
                "커버리지": coverage
            })
        except Exception as e:
            monthly_rows.append({
                "파일명": fp.name,
                "원본행수": np.nan,
                "전처리후행수": np.nan,
                "커버리지": np.nan,
                "오류": str(e)
            })

    monthly_summary = pd.DataFrame(monthly_rows)
    monthly_scores = pd.concat(monthly_scores_all, ignore_index=True) if monthly_scores_all else pd.DataFrame()

    if len(monthly_scores):
        yearly_combined = monthly_scores.groupby(["연도", "자치구코드", "모형"], as_index=False).agg(
            연평균_스트레스점수=("출퇴근스트레스점수", "mean"),
            연합계_총이동인구수=("총이동인구수", "sum"),
            연평균_평균이동시간=("평균이동시간", "mean"),
            월개수=("월", "nunique")
        )
    else:
        yearly_combined = pd.DataFrame()

    monthly_summary.to_csv(output_dir / "monthly_summary.csv", index=False, encoding="utf-8-sig")
    monthly_scores.to_csv(output_dir / "monthly_scores_all.csv", index=False, encoding="utf-8-sig")
    yearly_combined.to_csv(output_dir / "yearly_combined.csv", index=False, encoding="utf-8-sig")

    print("[DONE] 월별 배치 실행 완료")
    print(f"[INFO] 대상 파일 수: {len(files)}")
    print(f"[INFO] 저장 경로: {output_dir}")

    return monthly_summary, monthly_scores, yearly_combined


# 실행
monthly_summary_df, monthly_scores_df, yearly_combined_df = run_monthly_folder(
    input_dir=INPUT_DIR,
    file_pattern="*.csv",
    output_dir=OUTPUT_DIR / "monthly_batch"
)

print("\n[월별 처리 요약]")
display(monthly_summary_df)
print("\n[연도 통합 결과 (문헌기반 상위 10)]")
display(yearly_combined_df[yearly_combined_df["모형"] == "문헌기반"].sort_values("연평균_스트레스점수", ascending=False).head(10))